# FAI Football CV — Tracking Pipeline (v0.1)

Turns a short clip of game film into **per-frame player tracking**, a **two-team split**, an optional **top-down field map**, and a **JSON export** in the exact shape the FAI Film Room can import.

This is the football-adapted, lightweight cousin of Roboflow's *Basketball AI* pipeline. It favors choices that actually run today with minimal setup:

| Stage | This notebook (v0.1) | Heavier upgrade (later) |
|---|---|---|
| Detect players | **RF-DETR** (COCO `person`) | Fine-tuned football detector (players in pads) |
| Track | **ByteTrack** | SAM2 segmentation tracking |
| Team split | **Jersey chroma (LAB) + K-means** | SigLIP embeddings + UMAP + K-means |
| Field map | **Manual 4-point homography** | Trained field-keypoint model |
| Jersey numbers | *skipped in v0.1* | SmolVLM2 OCR (hardest on HS film) |
| Ball tracking | **not included** | — |

### Honest expectations
- Works best on **stable, wide sideline film**. Heavy pan/zoom, end-zone piles, and overlapping bodies degrade tracking — that's the real challenge, and why v0.1 is a *proof on your film* before anyone invests in fine-tuning.
- RF-DETR detects **people**, not football players specifically — refs, sideline players, and chain crew can get picked up. You'll filter most of that with the field homography.
- **No ball tracking** — a brown ball under stadium lights is near-impossible to track, and formation/tendency scouting is about where *players* line up and move, not the ball. Intentionally omitted.
- **Night film is the hard case.** Low light + stadium glare wash out jersey colors and add motion blur. This notebook fights that with an optional low-light boost (CLAHE) and a brightness-independent team split, but expect night HS film to be noisier than a daytime clip — run the proof on a real night game.
- No GPU = very slow. Set the Colab runtime to GPU first (next cell).

## 0. Setup
In Colab: **Runtime ▸ Change runtime type ▸ GPU**. Then run the install cell (~1–2 min).

In [ ]:
# One-time install. Pinned loosely; if an API changed, pin versions.
!pip -q install rfdetr supervision opencv-python-headless scikit-learn matplotlib
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Configure
Upload a short clip (10–30s is plenty for a proof) via the Colab Files panel, or mount Drive. Then set the path and options below.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    clip_path: str = 'clip.mp4'      # uploaded film clip
    angle: str = 'sideline'          # 'sideline' | 'endzone' (affects your homography points)
    conf: float = 0.4                # detection confidence (start lower for dark night film)
    low_light: bool = True           # CLAHE contrast boost before detection (helps night film)
    frame_stride: int = 3            # process every Nth frame (3 => ~10 fps from 30fps film)
    person_class_id: int = 1         # COCO 'person' id as returned by RF-DETR (adjust if needed)
    max_frames: int = 600            # safety cap so a long upload doesn't run forever
    out_json: str = 'fai_tracking.json'

CFG = Config()
print(CFG)

## 2. Detect players on one frame
Sanity check that detection works and that `person_class_id` is right before processing the whole clip.

In [ ]:
import cv2, numpy as np, supervision as sv
from rfdetr import RFDETRBase
from PIL import Image

model = RFDETRBase()

cap = cv2.VideoCapture(CFG.clip_path)
assert cap.isOpened(), f'Could not open {CFG.clip_path} — upload a clip and set CFG.clip_path'
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, frame0 = cap.read(); cap.release()
assert ok, 'Could not read first frame'
print(f'{W}x{H} @ {fps:.1f}fps')

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def enhance(bgr):
    # Low-light boost: lift contrast on the L (lightness) channel only, so
    # dark night frames get brighter without shifting jersey colors.
    if not CFG.low_light: return bgr
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = _clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

rgb = cv2.cvtColor(enhance(frame0), cv2.COLOR_BGR2RGB)
det = model.predict(Image.fromarray(rgb), threshold=CFG.conf)
people = det[det.class_id == CFG.person_class_id]
print('people detected on frame 0:', len(people))

box_ann = sv.BoxAnnotator()
annotated = box_ann.annotate(rgb.copy(), people)
sv.plot_image(annotated, size=(12, 7))

## 3. Detect + track across the clip (ByteTrack)
Stores one record per player per processed frame, plus a torso-color sample used for the team split in the next step.

In [ ]:
tracker = sv.ByteTrack(frame_rate=fps / CFG.frame_stride)

def torso_color(img_rgb, box):
    # Mean LAB chroma (a, b) of the jersey band — ignores the L/brightness
    # channel, so washed-out night jerseys still separate by hue.
    x1, y1, x2, y2 = [int(v) for v in box]
    h = y2 - y1
    ty1, ty2 = y1 + int(0.20 * h), y1 + int(0.55 * h)   # jersey band
    crop = img_rgb[max(ty1,0):max(ty2,1), max(x1,0):max(x2,1)]
    if crop.size == 0: return np.array([0.0, 0.0])
    lab = cv2.cvtColor(crop, cv2.COLOR_RGB2LAB)
    return lab[:, :, 1:3].reshape(-1, 2).mean(axis=0)   # (a, b)

records = []          # dicts: frame_idx, t, track_id, box(x1y1x2y2), foot(px)
track_colors = {}     # track_id -> list of torso colors

cap = cv2.VideoCapture(CFG.clip_path)
idx = 0; processed = 0
while processed < CFG.max_frames:
    ok, frame = cap.read()
    if not ok: break
    if idx % CFG.frame_stride != 0:
        idx += 1; continue
    rgb = cv2.cvtColor(enhance(frame), cv2.COLOR_BGR2RGB)
    det = model.predict(Image.fromarray(rgb), threshold=CFG.conf)
    det = det[det.class_id == CFG.person_class_id]
    det = tracker.update_with_detections(det)
    t = idx / fps
    for box, tid in zip(det.xyxy, det.tracker_id):
        if tid is None: continue
        x1, y1, x2, y2 = box
        foot = ((x1 + x2) / 2.0, y2)   # bottom-center = where the player stands
        records.append({'frame_idx': idx, 't': round(float(t), 3), 'track_id': int(tid),
                        'box': [float(x1), float(y1), float(x2), float(y2)],
                        'foot': [float(foot[0]), float(foot[1])]})
        track_colors.setdefault(int(tid), []).append(torso_color(rgb, box))
    processed += 1; idx += 1
cap.release()
print(f'processed {processed} frames, {len(records)} player-observations, {len(track_colors)} tracks')

## 4. Split into two teams (jersey chroma → K-means)
Clusters each *track's* average jersey **chroma** (LAB a/b, brightness removed) into two groups — so night jerseys that look dark still separate by hue. Good enough for contrasting kits; struggles when both teams wear similar colors (upgrade path: SigLIP embeddings).

In [ ]:
from sklearn.cluster import KMeans

tids = [t for t in track_colors if len(track_colors[t]) >= 2]
X = np.array([np.mean(track_colors[t], axis=0) for t in tids])
team_of = {}
if len(tids) >= 2:
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
    team_of = {tid: ('A' if lab == 0 else 'B') for tid, lab in zip(tids, labels)}
print('teams assigned:', len(team_of))

## 5. Map to the top-down field (click 4 points)
No coordinate typing. Run the cell, then **click 4 field points on the image** and it builds the homography for you.

**Click order:** near-left, near-right, far-left, far-right — i.e. two yard lines where each meets the near sideline, then where each meets the far sideline (the 4 corners of a box on the field).

Set `G` to how many **yards apart** your two lines are. Field coords are yards (length `x` 0–100, width `y` 0–53.3). Skip this cell to leave `field` null — image-space tracking still exports.

In [ ]:
# Click-to-calibrate: click 4 points on the image, get the homography. No typing coords.
import base64, json
from google.colab.output import eval_js
from IPython.display import display, Javascript

G = 10   # yards between the two yard lines you will click
FIELD_PTS = [(0, 0), (G, 0), (0, 53.3), (G, 53.3)]

_, _buf = cv2.imencode('.png', frame0)
_b64 = base64.b64encode(_buf).decode()

_js_template = '''
async function pick(n) {
  return await new Promise((resolve) => {
    const img = new Image();
    img.onload = () => {
      const info = document.createElement('div');
      info.textContent = 'Click ' + n + ' points: near-left, near-right, far-left, far-right';
      info.style = 'font:16px sans-serif;margin:6px 0;color:#111;background:#c6f24e;padding:4px';
      const c = document.createElement('canvas');
      c.width = img.width; c.height = img.height; c.style.maxWidth = '100%';
      c.getContext('2d').drawImage(img, 0, 0);
      document.body.appendChild(info); document.body.appendChild(c);
      const ctx = c.getContext('2d'); const pts = [];
      c.addEventListener('click', (e) => {
        const r = c.getBoundingClientRect();
        const x = Math.round((e.clientX - r.left) * img.width / r.width);
        const y = Math.round((e.clientY - r.top) * img.height / r.height);
        pts.push([x, y]);
        ctx.fillStyle = 'red'; ctx.beginPath(); ctx.arc(x, y, 9, 0, 7); ctx.fill();
        ctx.fillStyle = 'yellow'; ctx.font = 'bold 28px sans-serif'; ctx.fillText(pts.length, x + 12, y);
        if (pts.length >= n) resolve(JSON.stringify(pts));
      });
    };
    img.src = 'IMG_SRC';
  });
}
'''
_js = _js_template.replace('IMG_SRC', 'data:image/png;base64,' + _b64)
display(Javascript(_js))
IMG_PTS = json.loads(eval_js('pick(4)'))
H, _ = cv2.findHomography(np.array(IMG_PTS, np.float32), np.array(FIELD_PTS, np.float32))
print('clicked:', IMG_PTS)
print('homography ready' if H is not None else 'need 4 points')

def to_field(px, py):
    if H is None: return None
    p = np.array([[[px, py]]], np.float32)
    fx, fy = cv2.perspectiveTransform(p, H)[0][0]
    return [round(float(fx), 2), round(float(fy), 2)]

## 6. Preview the top-down radar
Scatter of one frame's players on a field rectangle, colored by team. Only meaningful once the homography is set.

In [ ]:
sample_t = records[len(records)//2]['t'] if records else 0
pts = [r for r in records if abs(r['t'] - sample_t) < 1e-6]
plt.figure(figsize=(11,6))
plt.gca().add_patch(plt.Rectangle((0,0),100,53.3,fill=False))
for yl in range(10,100,10): plt.plot([yl,yl],[0,53.3],color='0.85',lw=1)
for r in pts:
    f = to_field(*r['foot'])
    if f is None: continue
    c = {'A':'tab:red','B':'tab:blue'}.get(team_of.get(r['track_id']), 'gray')
    plt.scatter(f[0], f[1], c=c, s=60)
plt.xlim(-5,105); plt.ylim(-5,58); plt.title(f'Top-down @ t={sample_t:.2f}s'); plt.show()

## 7. Export JSON for the FAI Film Room
This is the **contract** with the app-side importer: normalized image coords (0–1, matching the Film Room's overlay), optional field yards, team, and track id. One entry per player per frame, grouped by frame.

In [ ]:
from collections import defaultdict
by_t = defaultdict(list)
for r in records:
    fx, fy = r['foot']
    by_t[r['t']].append({
        'trackId': r['track_id'],
        'team': team_of.get(r['track_id']),
        'number': None,                       # v0.1: jersey OCR not run
        'img': {'x': round(fx / W, 4), 'y': round(fy / H, 4)},   # normalized 0-1 to frame
        'field': to_field(fx, fy),            # yards or null
    })

out = {
    'meta': {'source': CFG.clip_path, 'fps': fps, 'angle': CFG.angle,
             'frameStride': CFG.frame_stride, 'createdWith': 'fai-football-cv v0.1'},
    'frames': [{'t': t, 'players': by_t[t]} for t in sorted(by_t)],
}
import json
with open(CFG.out_json, 'w') as f: json.dump(out, f)
print('wrote', CFG.out_json, '—', len(out['frames']), 'frames')
print(json.dumps(out['frames'][0], indent=2)[:600] if out['frames'] else 'no frames')

## 8. Limitations & where this goes next

**Known limits (v0.1)**
- Detects people, not football-specific players — expect refs/sideline noise until a football detector is fine-tuned.
- **Night film**: the CLAHE boost + LAB-chroma team split help, but low light and motion blur still cost detections and swap track IDs. If detections are sparse, lower `CFG.conf` and/or raise the CLAHE `clipLimit`. Fine-tuning a detector on your own night clips is the real fix.
- ByteTrack IDs swap through heavy occlusion (the snap, piles). SAM2 is the upgrade.
- Chroma team split still fails when both teams wear similar colors; SigLIP embeddings fix it.
- No jersey numbers yet (SmolVLM2 OCR is the hardest piece on HS film).
- No ball tracking (by design — impractical at night and not needed for formation/tendency scouting).
- Manual homography is per-camera; a trained field-keypoint model would automate it.

**How it feeds the app**
- `fai_tracking.json` uses the Film Room's normalized 0–1 image coords, so the planned importer can drop these straight in as **player tracks**, and use the field coords + clustering to **suggest a formation** (which a coach confirms) — feeding the tendency/scouting engine that already ships.

**Good next steps if the proof holds on your film**
1. Fine-tune RF-DETR on a football player dataset (Roboflow Universe has several) for clean player-only detection.
2. Swap ByteTrack → SAM2 for occlusion-robust tracking.
3. Build the FAI Film Room importer for this JSON (app-side, straightforward).